# hirid: WeavEHR → YAIB wide

This is the complete dataset workflow. It writes the all-concept WeavEHR wide table below the WeavEHR workspace `yaib/hirid/` directory. For RICU validation it additionally creates WeavEHR and RICU 7-day wide tables and comparison reports.


In [ ]:
import os
import subprocess
from pathlib import Path

from weavehr_yaib import (
    build_and_write_yaib_wide_for_dataset,
    compare_weavehr_wide_to_ricu_for_dataset,
    concept_root_from_output,
    resolve_weavehr_workspace,
    write_all_concepts_wide,
)

DATASET = "hirid"
MAX_HOURS = 7 * 24

# WeavEHR dataset version to use (e.g. "1.5.0"). None resolves it from the
# WeavEHR project and fails if several versions are present.
DATASET_VERSION = None

# Set this explicitly to the WeavEHR project workspace or another
# supported WeavEHR output location containing the concept output.
WEAVEHR_OUTPUT = None

if WEAVEHR_OUTPUT is None:
    raise ValueError("Set WEAVEHR_OUTPUT before running this notebook.")

WEAVEHR_OUTPUT = Path(WEAVEHR_OUTPUT).expanduser().resolve()
WORKSPACE = resolve_weavehr_workspace(WEAVEHR_OUTPUT)
CONCEPT_ROOT = concept_root_from_output(WEAVEHR_OUTPUT)
DATASET_OUTPUT = WORKSPACE / "yaib" / DATASET
DATASET_OUTPUT.mkdir(parents=True, exist_ok=True)

# Optional override if automatic raw stay-table discovery is not sufficient:
STAYS_PATH = None
RICU_CONCEPT_DICT = (
    Path.home() / "workspace" / "ricu" / "inst" / "extdata" / "config" / "concept-dict.json"
)

In [ ]:
# 1) Full WeavEHR YAIB-wide table across every available concept.
all_result = write_all_concepts_wide(
    dataset=DATASET,
    dataset_version=DATASET_VERSION,
    weavehr_output=WEAVEHR_OUTPUT,
    concept_root=CONCEPT_ROOT,
    stays_path=STAYS_PATH,
    max_hours=None,
    ricu_concept_dict=RICU_CONCEPT_DICT,
)
all_result

In [ ]:
# 2) Export the matching RICU reference files.
REPO_ROOT = Path.cwd().resolve()

env = os.environ.copy()
env["RICU_OUT_DIR"] = str(DATASET_OUTPUT)
# Declare the version of the RICU source data in RICU_DATA_PATH if known. Without
# it the reference version is only assumed (RICU config URL or manual default).
# env["RICU_SOURCE_VERSION"] = "..."
if "RICU_DATA_PATH" not in env:
    raise ValueError("Set RICU_DATA_PATH before running the RICU export.")
env["RICU_SRC_LOAD"] = "mimic,mimic_demo,eicu,eicu_demo,hirid,aumc,miiv,sic"
env.pop("RICU_CONFIG_PATH", None)
env["R_ENVIRON_USER"] = "/dev/null"

subprocess.run(
    [
        "Rscript",
        "--no-environ",
        "scripts/datasets/export_ricu_hirid.R",
    ],
    cwd=REPO_ROOT,
    check=True,
    env=env,
)

In [ ]:
# 3) WeavEHR YAIB-wide table for exactly the first 7 days.
weavehr_7d = build_and_write_yaib_wide_for_dataset(
    dataset=DATASET,
    dataset_version=DATASET_VERSION,
    max_hours=MAX_HOURS,
    output_root=DATASET_OUTPUT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=STAYS_PATH,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    version="1.0.0",
    aggregation_mode="ricu",
)
weavehr_7d

In [ ]:
# 4) Compare only the 7-day WeavEHR and RICU wide representations.
comparison = compare_weavehr_wide_to_ricu_for_dataset(
    dataset=DATASET,
    dataset_version=DATASET_VERSION,
    max_hours=MAX_HOURS,
    output_root=DATASET_OUTPUT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=STAYS_PATH,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    weavehr_wide_path=weavehr_7d.output_path,
)
comparison.as_dict()